# JSON -> Establish Generic Bronze-level Table

In [0]:
#Imports to run API Data Scraping script

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime
import logging
import traceback
import json
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import row_number
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pathlib import Path


In [0]:
#parameters that need to be changed between notebooks

bronze_level_schema = StructType([
    StructField('1711', StringType(), True),
    StructField('3000', StringType(), True),
    StructField('3001', StringType(), True),
    StructField('3005', StringType(), True),
    StructField('3006', StringType(), True),
    StructField('3007', StringType(), True),
    StructField('3008', StringType(), True),
    StructField('3009', StringType(), True),
    StructField('3058', StringType(), True),
    StructField('3066', StringType(), True),
    StructField('3075', StringType(), True),
    StructField('3132', StringType(), True),
    StructField('3144', StringType(), True),
    StructField('3174', StringType(), True),
    StructField('3256', StringType(), True),
    StructField('3260', StringType(), True),
    StructField('3359', StringType(), True),
    StructField('3387', StringType(), True),
    StructField('3389', StringType(), True),
    StructField('3390', StringType(), True),
    StructField('3393', StringType(), True),
    StructField('3395', StringType(), True),
    StructField('3405', StringType(), True),
    StructField('3504', StringType(), True),
    StructField('3509', StringType(), True),
    StructField('3596', StringType(), True),
    StructField('3640', StringType(), True),
    StructField('3684', StringType(), True),
    StructField('3722', StringType(), True),
    StructField('3730', StringType(), True),
    StructField('3771', StringType(), True),
    StructField('3775', StringType(), True),
    StructField('3780', StringType(), True),
    StructField('4111', StringType(), True),
    StructField('4112', StringType(), True),
    StructField('4121', StringType(), True),
    StructField('4131', StringType(), True),
    StructField('4214', StringType(), True),
    StructField('4411', StringType(), True),
    StructField('4511', StringType(), True),
    StructField('4722', StringType(), True),
    StructField('4784', StringType(), True),
    StructField('4814', StringType(), True),
    StructField('4829', StringType(), True),
    StructField('4899', StringType(), True),
    StructField('4900', StringType(), True),
    StructField('5045', StringType(), True),
    StructField('5094', StringType(), True),
    StructField('5192', StringType(), True),
    StructField('5193', StringType(), True),
    StructField('5211', StringType(), True),
    StructField('5251', StringType(), True),
    StructField('5261', StringType(), True),
    StructField('5300', StringType(), True),
    StructField('5310', StringType(), True),
    StructField('5311', StringType(), True),
    StructField('5411', StringType(), True),
    StructField('5499', StringType(), True),
    StructField('5533', StringType(), True),
    StructField('5541', StringType(), True),
    StructField('5621', StringType(), True),
    StructField('5651', StringType(), True),
    StructField('5655', StringType(), True),
    StructField('5661', StringType(), True),
    StructField('5712', StringType(), True),
    StructField('5719', StringType(), True),
    StructField('5722', StringType(), True),
    StructField('5732', StringType(), True),
    StructField('5733', StringType(), True),
    StructField('5812', StringType(), True),
    StructField('5813', StringType(), True),
    StructField('5814', StringType(), True),
    StructField('5815', StringType(), True),
    StructField('5816', StringType(), True),
    StructField('5912', StringType(), True),
    StructField('5921', StringType(), True),
    StructField('5932', StringType(), True),
    StructField('5941', StringType(), True),
    StructField('5942', StringType(), True),
    StructField('5947', StringType(), True),
    StructField('5970', StringType(), True),
    StructField('5977', StringType(), True),
    StructField('6300', StringType(), True),
    StructField('7011', StringType(), True),
    StructField('7210', StringType(), True),
    StructField('7230', StringType(), True),
    StructField('7276', StringType(), True),
    StructField('7349', StringType(), True),
    StructField('7393', StringType(), True),
    StructField('7531', StringType(), True),
    StructField('7538', StringType(), True),
    StructField('7542', StringType(), True),
    StructField('7549', StringType(), True),
    StructField('7801', StringType(), True),
    StructField('7802', StringType(), True),
    StructField('7832', StringType(), True),
    StructField('7922', StringType(), True),
    StructField('7995', StringType(), True),
    StructField('7996', StringType(), True),
    StructField('8011', StringType(), True),
    StructField('8021', StringType(), True),
    StructField('8041', StringType(), True),
    StructField('8043', StringType(), True),
    StructField('8049', StringType(), True),
    StructField('8062', StringType(), True),
    StructField('8099', StringType(), True),
    StructField('8111', StringType(), True),
    StructField('8931', StringType(), True),
    StructField('9402', StringType(), True),
])
schema = bronze_level_schema
schema_path = 'financial_transactions_dataset.default'
volume_path = '/Volumes/financial_transactions_dataset/default/financial_transactions_dataset_analytics_volume'
volume_identifier = 'mcc_codes'
table_name = 'bronze_mcc_codes_data'

In [0]:
#Set up logger to track pipeline activities

def setup_logging():

    # Ensure logs directory exists before starting logging
    os.makedirs('logs', exist_ok=True)

    logging.basicConfig(
        level = logging.INFO,
        format = '%(asctime)s - %(levelname)s - %(message)s',
        handlers = [logging.FileHandler(f'logs/bronze_level_table_run_{datetime.now().strftime('%Y%m%d')}.log'),
                    logging.StreamHandler(sys.stdout)
                   ]
    )

    return logging.getLogger(__name__)

In [0]:
#Check if a table exists in a schema; create an empty table if it does not exist.

def check_or_create_table(logger, table_name, schema_path):

    logger.info(f'Checking to see if {table_name} exists.')
    
    if spark.catalog.tableExists(f'{schema_path}.{table_name}'):
        logger.info("Table exists; table creation skipped.")
        flag = 1
    else:
        logger.info('Table does not exist; table will be created.')
        flag = 0
    
    return flag


In [0]:
#Function to identify correct files to load into bronze level table

def get_correct_files(logger, volume_path, volume_identifier):
    volume = []
    files_count = 0
    for vol_path,_,filenames in os.walk(volume_path):
        for file in filenames:
            if volume_identifier in file:
                files_count += 1
                logger.info(f'{file} is in the volume folder')
                volume.append(os.path.join(vol_path, file))
            else:
                pass
    logger.info(f'{files_count} files to load')
    return volume

In [0]:
#Function to read a csv data file for the bronze level table

def csv_data_extractor(logger, table_schema_bronze, updated_volume_paths):
    
    full_df = spark.createDataFrame([], table_schema_bronze)

    for file in updated_volume_paths:

        logger.info(f'Extracting data from {file}')

        df = spark.read.csv(file, header=True, sep = ',')
        total_row_count = df.count()
        total_column_count = len(df.columns)

        logger.info(f'Extracted {total_row_count} rows with {total_column_count} columns')
        
        full_df = full_df.union(df)
    
    logger.info(f'Extracted total {full_df.count()} rows with {len(full_df.columns)} columns')

    return full_df


In [0]:
#Function to read a json data file for the bronze level table

def json_data_extractor(logger, table_schema_bronze, updated_volume_path,):
    
    full_df = spark.createDataFrame([], table_schema_bronze)

    for file in updated_volume_path:

        logger.info(f'Extracting data from {file}')

        df = spark.read.option('multiline', True).json(file, schema = table_schema_bronze)
        total_row_count = df.count()
        total_column_count = len(df.columns)

        logger.info(f'Extracted {total_row_count} rows with {total_column_count} columns')
        
        full_df = full_df.union(df)

    return full_df


In [0]:
#Function to overwrite/append data to bronze level table

def create_or_append_to_table(logger, table_exists, df, schema_path, table_name):

    if table_exists == 1:
        logger.info(f'Appending data to {schema_path}')

        df.write.format('delta').mode('append').saveAsTable(f'{schema_path}.{table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    else:

        logger.info(f'Creating table to {schema_path}')

        df.write.format('delta').mode('overwrite').saveAsTable(f'{schema_path}.{table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    return


In [0]:
def bronze_level_ingestion(bronze_level_schema, schema_path, table_name, volume_path, volume_identifier):

    schema = bronze_level_schema
    schema_path = schema_path
    volume_path = volume_path
    volume_identifier = volume_identifier
    table_name = table_name
    
    logger = setup_logging()

    is_created = check_or_create_table(logger, table_name, schema_path)
    updated_volume_paths = get_correct_files(logger, volume_path, volume_identifier)
    df = json_data_extractor(logger, schema, updated_volume_paths)
    create_or_append_to_table(logger, is_created, df, schema_path, table_name)
    
    logger.info(f'Bronze level table {table_name} created or appended to successfully.')


In [0]:
bronze_level_ingestion(bronze_level_schema, schema_path, table_name, volume_path, volume_identifier)